# Exercise 7 - LLM Agent Simulations
_By Georg Ahnert, partially based on the course [IE 686 Large Language Models and Agents](https://www.uni-mannheim.de/dws/teaching/course-details/courses-for-master-candidates/ie-686-large-language-models-and-agents/)._

In this exercise you will use the [AutoGen](https://microsoft.github.io/autogen/stable/index.html) framework to simulate LLM agents.

**You will likely need to run this notebook on the BWUniCluster3.0 or on Google Colab to have enough GPU memory and compute.**

This exercise consists of three parts:
1. Setup
2. Agents and Tools
3. Teams of Agents

## 1. Setting up vllm and AutoGen

Follow the instructions from Exercise 3 for environment setup. Make sure you are using the correct kernel is selected for this notebook.

In [ ]:
# Install vllm to serve a model
%pip install vllm

In [ ]:
# Additionally, install AutoGen AgentChat and OpenAI client
%pip install "autogen-agentchat" "autogen-ext[openai]"

### Serve a local model with vllm

To use a locally hosted LLM with AutoGen, we need to serve the LLM in the background and connect AutoGen to it using the API. The API mostly follows [OpenAI's API specifications](https://platform.openai.com/docs/api-reference/chat?lang=curl).
We will also need our LLM to have the ability for [tool calling](https://docs.vllm.ai/en/latest/features/tool_calling.html) (see step 4 / 5). This allows us to define Python functions that the LLMs can interact with.

To serve a local LLM with vllm, follow these steps:

1. Open a new terminal (bottom left of the page on Colab)
2. **Only on the cluster:** Activate the Python virtual environment we have created in Exercise 3: `source llm4ess_env/bin/activate`
3. Make sure that vllm and AutoGen are installed (execute the cells above)
4. Start vllm as an [OpenAI-compatible server](https://docs.vllm.ai/en/latest/getting_started/quickstart.html#openai-compatible-server): `vllm serve "Qwen/Qwen3-4B-Instruct-2507" --enable-auto-tool-choice --tool-call-parser hermes`
5. **Only on Google Colab:** Since the T4 GPU has less memory available, we need to restrict the model length, so run this instead: `vllm serve "Qwen/Qwen3-4B-Instruct-2507" --max_model_len 30000 --enable-auto-tool-choice --tool-call-parser hermes`
6. Leave the terminal open to keep the server running

vllm will take some time to start up. Wait until the terminal says `INFO:     Application startup complete.`, then test whether you can reach the model like so:

In [ ]:
!curl http://localhost:8000/v1/models

### What is AutoGen?

AutoGen is an open-source framework developed by Microsoft Research that enables the creation and deployment of conversational AI agents that can collaborate with each other and with humans to solve complex tasks. It provides a flexible and customizable way to build multi-agent systems powered by LLMs.

### Key Features of AutoGen

- **Multi-Agent Conversations**: Create multiple agents that can communicate with each other to solve problems collaboratively.
- **Human-in-the-Loop**: Seamless integration of human feedback and oversight in agent workflows.
- **Customizable Agents**: Define agents with different personalities, skills, and roles.
- **Tool Use**: Agents can use external tools and APIs to expand their capabilities.
- **Memory and Context Management**: Sophisticated handling of conversation history and context.
- **Code Generation and Execution**: Built-in support for generating and running code, especially useful for data analysis and programming tasks.

Let's dive into some practical examples to see AutoGen in action!

### Using LLMs with AutoGen

In AutoGen, agents need access to Large Language Models (LLMs) to function. AutoGen provides a flexible way to connect to various model providers through model clients. 

AutoGen implements a protocol for model clients in `autogen-core` and provides implementations for popular model services in `autogen-ext`. These clients handle the communication between your agents and the underlying LLM services.

Let's connect AutoGen to our locally hosted model:

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import UserMessage, ModelInfo

# Specify the capabilities of our model
# AutoGen already has this information for some popular / closed-source OpenAI models
model_info=ModelInfo(
    vision=False,
    function_calling=True,
    json_output=True,
    family='unknown',
    structured_output=True,
    multiple_system_messages=False,
)

# Create an OpenAI model client for the locally hosted model
model_client = OpenAIChatCompletionClient(
    model="Qwen/Qwen3-4B-Instruct-2507", # the model we want to interact with
    base_url="http://localhost:8000/v1", # the address of our locally running server
    api_key="", # keys are only required for external providers
    model_info=model_info,
    temperature=0.7, # we can also specify decoding parameters
    seed=42,
)

model_client

In [ ]:
# Test the model client
async def test_openai():
    result = await model_client.create([
        UserMessage(content="What is AutoGen and how can it be used for multi-agent systems?", source="user")
    ])
    return result
    
result = await test_openai()
result.content[:1000]

In [ ]:
#Let's examine the result object in some more detail.
print("Finish reason:", result.finish_reason)
print("Usage:", result.usage)
print("Cached:", result.cached)
print("Logs:", result.logprobs)
print("Thinking:", result.thought)

## 2. Agents and Tools

It's important to understand the distinction between agents and models in AutoGen:

### Models
- **What they are**: The underlying LLMs (like GPT-4, Claude, Llama) that generate text
- **Function**: Process inputs and generate outputs based on their training
- **Role**: Provide the "brain" or reasoning capability
- **Implementation**: Accessed through model clients (connectors to API services or local deployments)

### Agents
- **What they are**: Autonomous entities with specific roles, skills, and behaviors
- **Function**: Coordinate actions, maintain context, interact with other agents/humans
- **Role**: Provide structure, persistence, and orchestration in multi-agent systems
- **Implementation**: Higher-level constructs that use models but also add:
  - Memory and conversation management
  - Tool usage capabilities
  - Specialized behaviors (coding, planning, etc.)
  - Workflow orchestration

Think of models as the cognitive engine, while agents are the full entities that use these engines to accomplish tasks within a broader system.

### AssistantAgent: The Workhorse of AutoGen

The `AssistantAgent` is a versatile agent that can use language models to generate responses and invoke tools to perform complex tasks. Let's see how to create and use an AssistantAgent:

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken

# Create an assistant agent
assistant = AssistantAgent(
    name="research_assistant",
    model_client=model_client,
    system_message="You are a helpful assistant who uses tools to find accurate information.",
)

# Test the agent with a simple query
response = await assistant.on_messages(
    [TextMessage(content="Tell me about AutoGen framework", source="user")],
    cancellation_token=CancellationToken(),
)

# Display the response
print("Final response:")
print(response.chat_message.content[:1000])

### Tool calling

Tools allow us to provide specific capabilities to the agents. They can be defined as Python functions that the agent can decide to call before it returns a reponse:

In [ ]:
# Define a simple tool that searches for information
async def web_search(query: str) -> str:
    """Find information on the web"""
    # In a real application, this would call a search API
    if "autogen" in query.lower():
        return "AutoGen is a programming framework for building multi-agent applications with LLMs."
    elif "university of mannheim" in query.lower():
        return "The University of Mannheim (German: Universität Mannheim), abbreviated UMA, is a public research university in Mannheim, Baden-Württemberg, Germany. Founded in 1967, the university has its origins in the Palatine Academy of Sciences, which was established by Elector Carl Theodor at Mannheim Palace in 1763, as well as the Handelshochschule (Commercial College Mannheim), which was founded in 1907."
    else:
        return f"Here are search results for: {query}"

# Create an assistant agent with the tool
assistant = AssistantAgent(
    name="research_assistant",
    model_client=model_client,
    tools=[web_search],
    system_message="You are a helpful assistant who uses tools to find accurate information. Always use the web_search tool when asked about factual information.",
)

# Test the agent with a simple query
response = await assistant.on_messages(
    [TextMessage(content="Tell me about AutoGen framework", source="user")],
    cancellation_token=CancellationToken(),
)

# Display the response
print("Final response:")
print(response.chat_message.content)

# You can also access the inner "thought process" messages
print("\nInner messages (thought process):")
for msg in response.inner_messages:
    print(f"- {msg.type}: {msg.content}")

### Streaming Responses for Better User Experience

AutoGen supports streaming responses, which provides a more interactive experience by showing the agent's work in real-time:

In [ ]:
from autogen_agentchat.ui import Console

# Stream the agent's responses to the console
await Console(
    assistant.on_messages_stream(
        [TextMessage(content="When was the University of Mannheim founded? Use your web_search tool.", source="user")],
        cancellation_token=CancellationToken(),
    ),
    output_stats=True,  # Show token usage statistics
)
print()

### The Agent Lifecycle

When you call `on_messages()` or `on_messages_stream()`:

1. The agent receives the input messages
2. Updates its internal state (memory)
3. Uses its model client to generate a response
4. If the model wants to use a tool, the agent:
   - Makes the tool call
   - Receives the tool result
   - (Optional) Reflects on the tool result
   - Generates a final response
5. Returns the final response

### Using Multiple Tools

Agents can use multiple tools to solve complex tasks:

In [ ]:
# Define additional tools
async def calculator(expression: str) -> str:
    """Calculate the result of a mathematical expression"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

async def current_date() -> str: # Note that this tool does not require a parameter to be passed
    """Get the current date"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d")

# Create an agent with multiple tools
multi_tool_assistant = AssistantAgent(
    name="multi_tool_assistant",
    model_client=model_client,
    tools=[calculator, current_date],
    system_message="You are a helpful assistant with access to multiple tools.",
)

# Test the agent with a task that requires multiple tools
await Console(
    multi_tool_assistant.on_messages_stream(
        [TextMessage(content="What's 342 * 15? Also, tell me today's date.", source="user")],
        cancellation_token=CancellationToken(),
    )
)

## Task 1: Simulate a social media response

Create a tool that uses the `model_client` to generate responses to social media posts. Then create a `social_media_agent` which uses this tool to respond to the following social media message: "Hello World"

## 3. Working with Teams

Teams in AutoGen allow multiple agents to collaborate on complex tasks. A team is a group of agents working together, each with their own specialization, to achieve a common goal through structured interaction.

Teams can also be used to simulate interactions of humans or (more generally) agents in an agent-based model (ABM).

### Basic Team Components

1. **Agents**: Individual agents with specialized roles
2. **Group Chat**: The conversation framework (like RoundRobinGroupChat)
3. **Termination Conditions**: Rules that determine when the team should stop

### Creating a Team for Project Proposal Development

Let's create a team for developing a project proposal for an LLM-based poker system. We'll use three agents:

1. **Idea Generator**: Proposes initial concepts for the poker system
2. **Critic**: Provides constructive criticism to find flaws and improvement areas
3. **Proposal Writer**: Refines the ideas incorporating feedback to create a polished proposal



In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken

# Create our specialized agents
idea_generator = AssistantAgent(
    name="idea_generator",
    model_client=model_client,
    system_message="""You are a creative AI researcher specialized in LLM applications and poker game development.
    Your role is to propose innovative ideas for an LLM-based poker system for the "Large Language Models and Agents" course.
    Focus on how LLMs can enhance poker gameplay, strategy analysis, or learning experiences.
    Be specific about technical implementations and the role of LLMs in your proposed system, but keep it short and don't write more than a couple of paragraphs.
    """
)

critic = AssistantAgent(
    name="critic",
    model_client=model_client,
    system_message="""You are a critical AI researcher with expertise in LLMs, game theory, and poker.
    Your role is to critically evaluate proposed poker system ideas, highlighting:
    1. Technical feasibility concerns within the course timeframe
    2. Potential challenges in implementation
    3. Limitations of current LLM capabilities in this context
    4. Areas where the proposal could be more specific or innovative
    Be constructive but thorough in identifying weaknesses, but keep it short and don't write more than a couple of paragraphs.
    """
)

proposal_writer = AssistantAgent(
    name="proposal_writer",
    model_client=model_client,
    system_message="""You are an expert proposal writer specialized in LLM applications.
    Your role is to synthesize ideas and critiques into coherent project proposals.
    Create structured, compelling proposals that:
    1. Integrate the original ideas with the critic's feedback
    2. Present a clear project scope, objectives, and implementation plan
    3. Highlight technical requirements and methodologies
    4. Address potential challenges proactively
    When you've created a final, refined proposal after multiple iterations, include the text "FINAL_PROPOSAL_COMPLETE".
    Don't just use what the idea_generator and the critic have come up with, but refine the proposal and provide an executive summary.
    """
)

# Create a termination condition - the team will stop when the proposal is complete
termination_condition = TextMentionTermination("FINAL_PROPOSAL_COMPLETE")

# Create the team with our agents in the desired order of conversation
proposal_team = RoundRobinGroupChat(
    participants=[idea_generator, critic, proposal_writer],
    termination_condition=termination_condition
)

### Running the Team

Now let's run our team to create a poker system proposal:

In [ ]:
# Define our task
poker_project_task = """
Create a comprehensive project proposal for an LLM-based poker system for the "Large Language Models and Agents" course.
The project should demonstrate innovative use of LLMs in the context of poker, 
potentially incorporating elements like strategy recommendation, opponent modeling, teaching, or game narration.
The proposal should be suitable for a team of 3-4 students to implement over 8 weeks.
"""

# Run the team with streaming to see the process in real-time
results = await Console(
    proposal_team.run_stream(task=poker_project_task),
    output_stats=True
)

## Task 2: Simulate a political discussion

For a political topic of you choice, simulate a discussion between a moderator and agents with different personas. Try out different types of [AutoGen Teams](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/teams.html) for the simulation.